In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/dataset.csv")
severity = pd.read_csv("../data/raw/Symptom-severity.csv")
description = pd.read_csv("../data/raw/symptom_Description.csv")
precaution = pd.read_csv("../data/raw/symptom_precaution.csv")

In [2]:
df.head()

,Disease,Symptom_1,Symptom_2,Symptom_3,Symptom_4,Symptom_5,Symptom_6,Symptom_7,Symptom_8,Symptom_9,Symptom_10,Symptom_11,Symptom_12,Symptom_13,Symptom_14,Symptom_15,Symptom_16,Symptom_17
0,Fungal infection,itching,skin_rash,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Fungal infection,skin_rash,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Fungal infection,itching,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Fungal infection,itching,skin_rash,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Fungal infection,itching,skin_rash,nodal_skin_eruptions,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
print("Rows, columns:", df.shape)
print("Number of diseases:", df["Disease"].nunique())

Rows, columns: (4920, 18)
Number of diseases: 41


In [4]:
severity.head(10)

,Symptom,weight
0,itching,1
1,skin_rash,3
2,nodal_skin_eruptions,4
3,continuous_sneezing,4
4,shivering,5
5,chills,3
6,joint_pain,3
7,stomach_pain,5
8,acidity,3
9,ulcers_on_tongue,4


In [5]:
df.isnull().sum()

Disease          0
Symptom_1        0
Symptom_2        0
Symptom_3        0
Symptom_4      348
Symptom_5     1206
Symptom_6     1986
Symptom_7     2652
Symptom_8     2976
Symptom_9     3228
Symptom_10    3408
Symptom_11    3726
Symptom_12    4176
Symptom_13    4416
Symptom_14    4614
Symptom_15    4680
Symptom_16    4728
Symptom_17    4848
dtype: int64

In [6]:
df["Disease"].value_counts()

Disease
Fungal infection                           120
Allergy                                    120
GERD                                       120
Chronic cholestasis                        120
Drug Reaction                              120
Peptic ulcer diseae                        120
AIDS                                       120
Diabetes                                   120
Gastroenteritis                            120
Bronchial Asthma                           120
Hypertension                               120
Migraine                                   120
Cervical spondylosis                       120
Paralysis (brain hemorrhage)               120
Jaundice                                   120
Malaria                                    120
Chicken pox                                120
Dengue                                     120
Typhoid                                    120
hepatitis A                                120
Hepatitis B                                120
Hepat

In [7]:
print(repr(df.loc[0, "Symptom_1"]))
print(repr(df.loc[0, "Symptom_2"]))
print(df["Disease"].unique()[:10])

'itching'
' skin_rash'
<StringArray>
[   'Fungal infection',             'Allergy',                'GERD',
 'Chronic cholestasis',       'Drug Reaction', 'Peptic ulcer diseae',
                'AIDS',           'Diabetes ',     'Gastroenteritis',
    'Bronchial Asthma']
Length: 10, dtype: str


In [8]:
print("Duplicate rows:", df.duplicated().sum())
print("Unique rows:", len(df.drop_duplicates()))

Duplicate rows: 4616
Unique rows: 304


In [9]:
dataset_symptoms = set(pd.unique(df.iloc[:, 1:].values.ravel()))
dataset_symptoms = {str(s).strip() for s in dataset_symptoms if pd.notna(s)}
severity_symptoms = set(severity["Symptom"].str.strip())

print("In dataset but not in severity file:", dataset_symptoms - severity_symptoms)

In dataset but not in severity file: {'foul_smell_of urine', 'spotting_ urination', 'dischromic _patches'}


In [10]:
import re

def clean_text(s):
    if pd.isna(s):
        return s
    s = str(s).strip().lower()
    s = re.sub(r"[\s_]+", "_", s)   # turn spaces/underscores into one underscore
    return s

symptom_cols = df.columns[1:]
for col in symptom_cols:
    df[col] = df[col].apply(clean_text)

severity["Symptom"] = severity["Symptom"].apply(clean_text)

In [11]:
dataset_symptoms = {s for s in pd.unique(df[symptom_cols].values.ravel()) if pd.notna(s)}
severity_symptoms = set(severity["Symptom"])

print("In dataset, not in severity:", dataset_symptoms - severity_symptoms)
print("In severity, not in dataset:", severity_symptoms - dataset_symptoms)
print("Total unique symptoms:", len(dataset_symptoms))

In dataset, not in severity: {'foul_smell_of_urine'}
In severity, not in dataset: {'prognosis', 'foul_smell_ofurine'}
Total unique symptoms: 131


In [12]:
# Fix the typo and remove the junk row
severity["Symptom"] = severity["Symptom"].replace("foul_smell_ofurine", "foul_smell_of_urine")
severity = severity[severity["Symptom"] != "prognosis"]

# Check for duplicate symptoms in the severity file
print("Duplicate symptoms in severity file:")
print(severity[severity["Symptom"].duplicated(keep=False)])

Duplicate symptoms in severity file:
            Symptom  weight
45   fluid_overload       6
117  fluid_overload       4


In [13]:
# Keep the higher weight for duplicated symptoms (safer for triage)
severity = (severity.sort_values("weight", ascending=False)
                    .drop_duplicates(subset="Symptom")
                    .reset_index(drop=True))

print("Severity rows:", len(severity))
print("Any duplicates left:", severity["Symptom"].duplicated().any())

Severity rows: 131
Any duplicates left: False


In [14]:
dataset_symptoms = {s for s in pd.unique(df[symptom_cols].values.ravel()) if pd.notna(s)}
severity_symptoms = set(severity["Symptom"])

print("In dataset, not in severity:", dataset_symptoms - severity_symptoms)
print("In severity, not in dataset:", severity_symptoms - dataset_symptoms)
print("Total unique symptoms:", len(dataset_symptoms))

In dataset, not in severity: set()
In severity, not in dataset: set()
Total unique symptoms: 131


In [15]:
disease_fixes = {
    "Peptic ulcer diseae": "Peptic ulcer disease",
    "Osteoarthristis": "Osteoarthritis",
    "(vertigo) Paroymsal Positional Vertigo": "Paroxysmal Positional Vertigo",
    "hepatitis A": "Hepatitis A",
    "Dimorphic hemmorhoids(piles)": "Hemorrhoids (piles)",
}

def clean_disease(name):
    name = re.sub(r"\s+", " ", str(name).strip())
    return disease_fixes.get(name, name)

df["Disease"] = df["Disease"].apply(clean_disease)
description["Disease"] = description["Disease"].apply(clean_disease)
precaution["Disease"] = precaution["Disease"].apply(clean_disease)

print(sorted(df["Disease"].unique()))

['AIDS', 'Acne', 'Alcoholic hepatitis', 'Allergy', 'Arthritis', 'Bronchial Asthma', 'Cervical spondylosis', 'Chicken pox', 'Chronic cholestasis', 'Common Cold', 'Dengue', 'Diabetes', 'Drug Reaction', 'Fungal infection', 'GERD', 'Gastroenteritis', 'Heart attack', 'Hemorrhoids (piles)', 'Hepatitis A', 'Hepatitis B', 'Hepatitis C', 'Hepatitis D', 'Hepatitis E', 'Hypertension', 'Hyperthyroidism', 'Hypoglycemia', 'Hypothyroidism', 'Impetigo', 'Jaundice', 'Malaria', 'Migraine', 'Osteoarthritis', 'Paralysis (brain hemorrhage)', 'Paroxysmal Positional Vertigo', 'Peptic ulcer disease', 'Pneumonia', 'Psoriasis', 'Tuberculosis', 'Typhoid', 'Urinary tract infection', 'Varicose veins']


In [16]:
diseases = set(df["Disease"])
print("Missing from description file:", diseases - set(description["Disease"]))
print("Missing from precaution file:", diseases - set(precaution["Disease"]))

Missing from description file: {'Hemorrhoids (piles)'}
Missing from precaution file: set()


In [17]:
df = df.drop_duplicates().reset_index(drop=True)
print("Rows after removing duplicates:", len(df))
print(df["Disease"].value_counts().describe())

Rows after removing duplicates: 304
count    41.000000
mean      7.414634
std       1.702581
min       5.000000
25%       6.000000
50%       7.000000
75%       9.000000
max      10.000000
Name: count, dtype: float64


In [18]:
print(description[description["Disease"].str.contains("piles|hemm|hemorr", case=False)])

                         Disease  \
16  Dimorphic hemorrhoids(piles)   
24  Paralysis (brain hemorrhage)   

                                          Description  
16  Hemorrhoids, also spelled haemorrhoids, are va...  
24  Intracerebral hemorrhage (ICH) is when blood s...  


In [19]:
description.loc[description["Disease"].str.contains("piles", case=False), "Disease"] = "Hemorrhoids (piles)"

print("Missing from description file:", diseases - set(description["Disease"]))

Missing from description file: set()


In [20]:
df.to_csv("../data/processed/cleaned_dataset.csv", index=False)
severity.to_csv("../data/processed/symptom_severity.csv", index=False)
description.to_csv("../data/processed/disease_description.csv", index=False)
precaution.to_csv("../data/processed/disease_precaution.csv", index=False)
print("Saved!")

Saved!
